# Distill DeepSeek-R1 -> Qwen3-0.6B — 3-seed run (500-article study)

Trains **3 tuned models** (seeds 42, 123, 7) on the 401-article train split, exports a `q4_k_m` GGUF per seed for Ollama. Three seeds → we report mean ± spread, not one lucky run.

**Runtime:** Colab → Change runtime type → **T4 GPU**.
**Before running:** upload `train_chat.jsonl` (from `data/eval/`) via the Files panel.

Each seed takes ~5–7 min on a T4; ~20 min total.

In [ ]:
!pip install -q unsloth

In [ ]:
import torch, gc
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

MAX_SEQ = 2048
SEEDS = [42, 123, 7]
dataset = load_dataset('json', data_files='train_chat.jsonl', split='train')
print('train examples:', len(dataset))

In [ ]:
def train_seed(seed):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/Qwen3-0.6B', max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=32,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=32, lora_dropout=0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=seed)

    def fmt(ex):
        return {'text': [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False, enable_thinking=False) for c in ex['messages']]}
    ds = dataset.map(fmt, batched=True)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ,
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
            logging_steps=10, optim='adamw_8bit', weight_decay=0.01,
            lr_scheduler_type='linear', seed=seed, output_dir=f'out_{seed}', report_to='none'))
    trainer = train_on_responses_only(trainer,
        instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
    trainer.train()

    out = f'qwen3-0.6b-rss-seed{seed}'
    model.save_pretrained_gguf(out, tokenizer, quantization_method='q4_k_m')
    print(f'=== seed {seed} done -> {out} ===')
    del model, trainer, ds; gc.collect(); torch.cuda.empty_cache()

for s in SEEDS:
    print(f'\n########## TRAINING SEED {s} ##########')
    train_seed(s)

In [ ]:
# List the exported GGUFs to download
!ls -lh qwen3-0.6b-rss-seed*/*.gguf

## Bring the 3 tuned models to Ollama on your Mac

Download each `*q4_k_m.gguf` (Files panel → download). For each seed, drop it in a folder with a `Modelfile`:

```
FROM ./qwen3-0.6b-rss-seed42.Q4_K_M.gguf
PARAMETER temperature 0
```

Then create all three:

```bash
ollama create rss-tuned-s42 -f Modelfile.s42
ollama create rss-tuned-s123 -f Modelfile.s123
ollama create rss-tuned-s7 -f Modelfile.s7
```

(Temperature 0 to match the temp-0 eval protocol.) The eval harness picks these up as the `tuned` arm's 3 seeds.